# Notebook 0: Plot Data 

This is the first notebook in the sequence for the HRS Botany project.

In it we will download, prepare, and merge plot data from the UCSC Forest Ecology Research Plot and the Open Forest Observatory.

The plot data just includes the plot metadata for the FERP plot and all of the OFO plots. We will use the merged plot dataset to determine which EnMAP images to download in a subsequent notebook. 

__Notebook Inputs:__
- FERP Data (_FERP123merged_20231029.csv_)
- FERP Tree Species Schema (_ferp_tree_species.txt_)

Notebook Outputs:
- Plot Dataset (_data/plots.csv_)


In [1]:
# Imports

import numpy as np
import pandas as pd

import geopandas as gpd
from shapely.geometry import Point

from hrs_botany.data_utils import clean_ofo_survey_date, load_ferp_species_table

ModuleNotFoundError: No module named 'hrs_botany'

## 1. FERP Data Acquisition

First we will prepare the OFO plot data. 

The OFO plot data is separate from the trees data, and has a single row for each plot. Each plot row includes many metadata fields, but for our purposes, the columns of interest are _plot_id_, _plot_area_ha_, _survey_date_, and 
_geometry_. 


In [18]:
ofo_plots_df = gpd.read_file('../../data/ofo/ofo_ground-reference_plots.gpkg')

In [20]:
ofo_plots_df = ofo_plots_df[['plot_id', 'plot_area_ha', 'survey_date', 'geometry']]

ofo_plots_df.head(5)

,plot_id,plot_area_ha,survey_date,geometry
0,0070,4.273060,2008.0,"MULTIPOLYGON (((-120.00103 38.17605, -120.0011..."
1,0081,4.000000,201910.0,"POLYGON ((-116.76897 33.81007, -116.76767 33.8..."
2,0083,4.003663,202008.0,"MULTIPOLYGON (((-123.86914 40.77879, -123.8691..."
3,0084,4.000000,201907.0,"MULTIPOLYGON (((-119.73617 37.09988, -119.7361..."
4,0087,3.979825,201906.0,"MULTIPOLYGON (((-119.02333 36.96424, -119.0210..."


In [21]:
ofo_plots_df.plot_area_ha.describe()

count    258.000000
mean       0.449594
std        0.864867
min        0.020106
25%        0.079862
50%        0.129533
75%        0.291864
max        4.273060
Name: plot_area_ha, dtype: float64

It is important to our study to restrict our plot selection to plots of a large enough area to cover several contiguous EnMAP pixels (30x30m). Otherwise, we will not be able to determine species absence in a full pixel. 

__We choose a minimum plot size of 0.5 hectare – this value needs a stronger scientific rationale, and should be revisited.__

In [22]:
ofo_plots_df = ofo_plots_df[ofo_plots_df.plot_area_ha > 0.5].reset_index(drop=True)

ofo_plots_df = ofo_plots_df[ofo_plots_df['plot_id'].notna()].reset_index(drop=True)

ofo_plots_df = ofo_plots_df.sort_values(by='plot_id').reset_index(drop=True)    

In [24]:
print('Number of OFO plots:', len(ofo_plots_df))

ofo_plots_df.plot_area_ha.describe()


Number of OFO plots: 33


count    33.000000
mean      2.414287
std       1.160409
min       0.785000
25%       1.393045
50%       2.522393
75%       3.459541
max       4.273060
Name: plot_area_ha, dtype: float64

In [27]:
ofo_plots_df.head(5)

,plot_id,plot_area_ha,survey_date,geometry
0,0058,1.463947,20200809.0,"POLYGON ((-121.11482 40.12925, -121.11517 40.1..."
1,0059,0.945688,20200715.0,"POLYGON ((-122.68571 38.81993, -122.68595 38.8..."
2,0060,1.308436,20200823.0,"POLYGON ((-123.56877 40.29993, -123.56915 40.2..."
3,0061,1.286581,20200704.0,"POLYGON ((-122.5369 41.03392, -122.53713 41.03..."
4,0062,1.398960,20210621.0,"POLYGON ((-123.56378 40.30005, -123.56385 40.3..."


In [ ]:
import pandas as pd



# —— Usage example ——
ofo_plots_df["survey_date_fixed"] = fix_survey_date(ofo_plots_df["survey_date"])

# (Optional) flag which dates were approximated (i.e. those that weren’t full YYYYMMDD)
ofo_plots_df["survey_date_approx"] = (
    ofo_plots_df["survey_date"]
    .astype("Int64")
    .astype(str)
    .str.len()
    .lt(8)
)


In [30]:
ofo_plots_df

,plot_id,plot_area_ha,survey_date,geometry,survey_date_fixed,survey_date_approx
0,0058,1.463947,20200809.0,"POLYGON ((-121.11482 40.12925, -121.11517 40.1...",2020-08-09,False
1,0059,0.945688,20200715.0,"POLYGON ((-122.68571 38.81993, -122.68595 38.8...",2020-07-15,False
2,0060,1.308436,20200823.0,"POLYGON ((-123.56877 40.29993, -123.56915 40.2...",2020-08-23,False
3,0061,1.286581,20200704.0,"POLYGON ((-122.5369 41.03392, -122.53713 41.03...",2020-07-04,False
4,0062,1.398960,20210621.0,"POLYGON ((-123.56378 40.30005, -123.56385 40.3...",2021-06-21,False
5,0063,1.462966,20200725.0,"POLYGON ((-122.5425 41.02278, -122.54262 41.02...",2020-07-25,False
6,0066,1.404556,20210829.0,"POLYGON ((-119.42135 37.23617, -119.42204 37.2...",2021-08-29,False
7,0067,1.393045,20210831.0,"POLYGON ((-119.41176 37.26544, -119.41177 37.2...",2021-08-31,False
8,0068,3.230183,20190915.0,"POLYGON ((-120.08843 38.96572, -120.08844 38.9...",2019-09-15,False
9,0069,3.459541,2016.0,"MULTIPOLYGON (((-120.01534 38.18483, -120.0177...",2016-01-01,True


In [28]:
ofo_plots_df.survey_date

0     20200809.0
1     20200715.0
2     20200823.0
3     20200704.0
4     20210621.0
5     20200725.0
6     20210829.0
7     20210831.0
8     20190915.0
9         2016.0
10        2008.0
11        2008.0
12        2007.0
13        2018.0
14        2018.0
15        2018.0
16        2018.0
17        2018.0
18        2018.0
19        2018.0
20        2018.0
21      201910.0
22      201907.0
23      202008.0
24      201907.0
25      202006.0
26      202006.0
27      201906.0
28    20230712.0
29    20220701.0
30    20220701.0
31    20220701.0
32    20220701.0
Name: survey_date, dtype: float64

## 2. FERP Data Acquisition

Now we will prepare the FERP plot.

Since FERP is just a single plot, it will just be a single row in our plot dataset.

I could not find a polygon file for the FERP plot, so we're doing it manually.

We'll load the FERP dataset, and create a rotated rectangular boundary for the plot geometry.

## 2. FERP Data Acquisition

In [3]:
ferp_path = '../../data/ferp/geoforest/doi_10_5061_dryad_6q573n64s__v20240129/FERP123merged_20231029.csv'

# Load the dataset
df = pd.read_csv(ferp_path)

# Create GeoDataFrame
df_geo = df.dropna(subset=["east_UTM", "north_UTM"]).copy()
gdf = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.points_from_xy(df_geo["east_UTM"], df_geo["north_UTM"]),
    crs="EPSG:32610"  # UTM Zone 10N
)

# Create rotated bounding box from stem points
ferp_boundary = gpd.GeoDataFrame(
    geometry=[gdf.unary_union.minimum_rotated_rectangle],
    crs=gdf.crs  # same CRS as stem points
)

/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_79407/233959029.py:4: DtypeWarning: Columns (13,16,17,23,25,27,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ferp_path)
/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_79407/233959029.py:16: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[gdf.unary_union.minimum_rotated_rectangle],
